# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,28.5,0.031194,-0.586957,0.206522,23,6
1,CTR vs PD,PES_RG,23.5,0.015367,-0.659420,0.170290,23,6
2,CTR vs PD,PES_BY,28.0,0.029204,-0.594203,0.202899,23,6
3,CTR vs PD,VKS_MajRad,30.0,0.035674,-0.565217,0.217391,23,6
4,CTR vs PD,VKS_MinRad,28.0,0.026348,-0.594203,0.202899,23,6
5,CTR vs PD,VKS_Angle,69.0,1.000000,0.000000,0.500000,23,6
6,HC vs protan,TES,6.0,0.000048,-0.942029,0.028986,23,9
7,HC vs protan,PES_RG,0.0,0.000016,-1.000000,0.000000,23,9
8,HC vs protan,PES_BY,46.5,0.017860,-0.550725,0.224638,23,9
9,HC vs protan,VKS_MajRad,7.0,0.000057,-0.932367,0.033816,23,9


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,6.0,0.000008,-0.934783,0.032609,23,8
7,HC vs protan,PES_RG,0.0,0.000016,-1.000000,0.000000,23,9
15,HC vs deutan,VKS_MajRad,8.0,0.000017,-0.913043,0.043478,23,8
13,HC vs deutan,PES_RG,0.0,0.000036,-1.000000,0.000000,23,8
6,HC vs protan,TES,6.0,0.000048,-0.942029,0.028986,23,9
9,HC vs protan,VKS_MajRad,7.0,0.000057,-0.932367,0.033816,23,9
12,HC vs deutan,TES,7.5,0.000148,-0.918478,0.040761,23,8
10,HC vs protan,VKS_MinRad,16.0,0.000266,-0.845411,0.077295,23,9
14,HC vs deutan,PES_BY,23.0,0.001983,-0.750000,0.125000,23,8
11,HC vs protan,VKS_Angle,170.0,0.005670,0.642512,0.821256,23,9


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
16,HC vs deutan,VKS_MinRad,6.0,0.000008,-0.934783,0.032609,23,8,0.000046,True
15,HC vs deutan,VKS_MajRad,8.0,0.000017,-0.913043,0.043478,23,8,0.000085,True
7,HC vs protan,PES_RG,0.0,0.000016,-1.000000,0.000000,23,9,0.000095,True
13,HC vs deutan,PES_RG,0.0,0.000036,-1.000000,0.000000,23,8,0.000144,True
9,HC vs protan,VKS_MajRad,7.0,0.000057,-0.932367,0.033816,23,9,0.000238,True
6,HC vs protan,TES,6.0,0.000048,-0.942029,0.028986,23,9,0.000238,True
12,HC vs deutan,TES,7.5,0.000148,-0.918478,0.040761,23,8,0.000445,True
10,HC vs protan,VKS_MinRad,16.0,0.000266,-0.845411,0.077295,23,9,0.000798,True
14,HC vs deutan,PES_BY,23.0,0.001983,-0.750000,0.125000,23,8,0.003967,True
11,HC vs protan,VKS_Angle,170.0,0.005670,0.642512,0.821256,23,9,0.011341,True
